# gVXR → TIGRE reconstruction validation (Colab)

This notebook reconstructs a clean, monochromatic gVXR scan with TIGRE FDK and compares the result with the saved `ground_truth/attenuation_cm_inv` volume. It reports per-material mean attenuation, RMSE, and Dice overlap.

Use a dataset file generated by the project’s HDF5 dataset generator (`sample_*.h5`). It must be a **monochromatic, ideal/clean scan**. The notebook uses `projection/attenuation` and saved metadata; it does not use raw counts.

TIGRE’s Python implementation is CUDA-based. In Colab, select **Runtime → Change runtime type → T4 GPU** before installing it. TIGRE installation compiles its Python/CUDA extension and can take several minutes. [TIGRE installation instructions](https://github.com/CERN/TIGRE/blob/master/Frontispiece/python_installation.md) · [geometry example](https://github.com/CERN/TIGRE/blob/master/Python/demos/d01_CreateGeometry.py).

> Coordinate conventions must be checked for your scan. This notebook starts with the saved gVXR angles and TIGRE’s standard cone geometry. It includes a flip/angle-sign comparison table; use an asymmetric object to select a convention and record it. A symmetric sphere alone cannot determine every flip.


## 1. Check the Colab GPU

In [ ]:
!nvidia-smi


If this cell says no NVIDIA GPU is available, change the Colab runtime to GPU and reconnect before continuing.

## 2. Install TIGRE

TIGRE’s official Python installation builds from its CERN repository. The install below uses the Colab runtime’s CUDA compiler/toolchain.

In [ ]:
%cd /content
!if [ ! -d /content/TIGRE/.git ]; then git clone --depth 1 https://github.com/CERN/TIGRE.git; fi
%cd /content/TIGRE
import os, shutil, subprocess, sys
from pathlib import Path

nvcc = shutil.which('nvcc')
if nvcc is None:
    candidates = sorted(Path('/usr/local').glob('cuda*/bin/nvcc'))
    nvcc = str(candidates[-1]) if candidates else None
if nvcc is None:
    raise RuntimeError('TIGRE source build requires the CUDA compiler (nvcc). Confirm Colab is using a GPU runtime; if nvcc is still missing, this runtime cannot compile TIGRE.')
cuda_home = str(Path(nvcc).resolve().parent.parent)
os.environ['CUDA_HOME'] = cuda_home
os.environ['CUDA_PATH'] = cuda_home
os.environ['PATH'] = str(Path(nvcc).resolve().parent) + os.pathsep + os.environ.get('PATH', '')
print('Using CUDA_HOME:', cuda_home)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'Cython', 'wheel', 'setuptools'])
# Build against the CUDA libraries available in this Colab runtime.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '-v', '.'])
%cd /content


In [ ]:
import numpy as np
import tigre
import tigre.algorithms as algs
print('TIGRE imported:', getattr(tigre, '__version__', 'version not exposed'))


## 3. Upload one clean HDF5 sample

In [ ]:
from google.colab import files
uploaded = files.upload()
h5_path = next((name for name in uploaded if name.lower().endswith('.h5')), None)
if h5_path is None:
    raise ValueError('Upload one sample_*.h5 file from the generated dataset ZIP.')
print('Using:', h5_path)


In [ ]:
import json, h5py

with h5py.File(h5_path, 'r') as h:
    required = ['projection/attenuation', 'angles_deg', 'ground_truth/labels',
                'ground_truth/attenuation_cm_inv']
    missing = [k for k in required if k not in h]
    if missing:
        raise KeyError(f'Missing HDF5 datasets: {missing}')
    proj = h['projection/attenuation'][:].astype(np.float32)
    angles_deg = h['angles_deg'][:].astype(np.float32)
    labels_xyz = h['ground_truth/labels'][:]
    truth_xyz = h['ground_truth/attenuation_cm_inv'][:].astype(np.float32)
    cfg = json.loads(h.attrs['scan_config_json'])
    gt_meta = json.loads(h.attrs['ground_truth_json'])
    effects = json.loads(h.attrs.get('effects_json', '{}'))

print('Projection array:', proj.shape, '(angle, detector row, detector column)')
print('Angles:', len(angles_deg), 'Ground truth xyz:', truth_xyz.shape)
print('Ground-truth metadata:', gt_meta)
print('Effects metadata:', effects)

if cfg.get('beam', 'Cone') != 'Cone':
    raise ValueError('This notebook sets cone-beam geometry; use a cone-beam sample.')
if cfg.get('spectrum') not in (None, {}, ''):
    raise ValueError('Choose a monochromatic sample (scan_config_json spectrum must be empty/null).')
if cfg.get('noise', False):
    raise ValueError('Choose a clean sample with scan_config_json noise=false.')
if effects and any(float(effects.get(k, 0) or 0) != 0 for k in
                   ['psf_sigma_px','scatter_frac','gain_sigma','flat_mismatch_sigma','read_noise','dark_offset']):
    raise ValueError('This sample has detector effects; choose the clean/no-effects variant.')
if effects.get('poisson', False):
    raise ValueError('This sample has Poisson noise; choose the clean/no-effects variant.')
if proj.ndim != 3 or proj.shape[0] != len(angles_deg):
    raise ValueError('Projection shape does not match angles_deg.')
if not np.isfinite(proj).all() or not np.isfinite(truth_xyz).all():
    raise ValueError('Input arrays contain NaN or infinite values.')


## 4. Build TIGRE cone geometry from saved metadata

Distances and detector dimensions follow the project’s `geometry.json` definitions. TIGRE geometry uses voxel order `(z, y, x)` for Python volumes; the stored ground truth is `(x, y, z)`, so it is transposed below for comparison. The GT origin is converted to a volume-center offset.


In [ ]:
SOD = float(cfg['source_x_cm']) * 10.0
SDD = (float(cfg['source_x_cm']) + float(cfg['detector_x_cm'])) * 10.0
px = float(cfg['pixel_mm'])
mag = SDD / SOD
voxel_mm = float(gt_meta['voxel_size_mm'])
voxel_expected_mm = px / mag

if not np.isclose(voxel_mm, voxel_expected_mm, rtol=1e-4, atol=1e-6):
    print(f'WARNING: GT voxel pitch {voxel_mm:g} mm differs from pixel/magnification {voxel_expected_mm:g} mm.')

ny, nx = int(cfg['ny']), int(cfg['nx'])
if proj.shape[1:] != (ny, nx):
    raise ValueError(f'Projection detector shape {proj.shape[1:]} != config {(ny, nx)}')

# TIGRE volumes use (z, y, x); saved ground truth uses (x, y, z).
nx_gt, ny_gt, nz_gt = truth_xyz.shape
geo = tigre.geometry()
geo.mode = 'cone'
geo.DSD, geo.DSO = SDD, SOD
geo.nDetector = np.array([ny, nx], dtype=np.int32)
geo.dDetector = np.array([px, px], dtype=np.float32)
geo.sDetector = geo.nDetector * geo.dDetector
geo.nVoxel = np.array([nz_gt, ny_gt, nx_gt], dtype=np.int32)
geo.dVoxel = np.array([voxel_mm, voxel_mm, voxel_mm], dtype=np.float32)
geo.sVoxel = geo.nVoxel * geo.dVoxel
geo.offDetector = np.array([0.0, 0.0], dtype=np.float32)
geo.rotDetector = np.array([0.0, 0.0, 0.0], dtype=np.float32)
geo.accuracy = 0.5

# Align the volume center with the saved ground-truth grid.
origin_xyz = np.asarray(gt_meta['origin_mm'], dtype=float)
shape_xyz = np.asarray(truth_xyz.shape, dtype=float)
volume_center_xyz = origin_xyz + 0.5 * shape_xyz * voxel_mm
geo.offOrigin = volume_center_xyz.astype(np.float32)

ang = np.deg2rad(angles_deg).astype(np.float32)
print(f'SOD={SOD:.3f} mm, SDD={SDD:.3f} mm, magnification={mag:.5f}')
print('TIGRE nVoxel (z,y,x):', geo.nVoxel, 'voxel pitch:', geo.dVoxel, 'mm')
print('TIGRE offOrigin (x,y,z) mm:', geo.offOrigin)


## 5. Reconstruct with FDK

FDK is the standard approximate filtered backprojection for circular cone-beam scans. TIGRE’s geometry fields and `fdk(projections, geo, angles)` follow the official Python examples.


In [ ]:
# Start with the saved angle sign and detector orientation.
recon_mm_inv = algs.fdk(proj, geo, ang).astype(np.float32)
print('FDK volume shape:', recon_mm_inv.shape, '(expected TIGRE z,y,x)')
print('FDK range (mm^-1):', float(np.nanmin(recon_mm_inv)), float(np.nanmax(recon_mm_inv)))

# Interpret FDK output as mm^-1.
recon_cm_inv = recon_mm_inv * 10.0
truth_zyx = np.transpose(truth_xyz, (2, 1, 0))
labels_zyx = np.transpose(labels_xyz, (2, 1, 0))
if recon_cm_inv.shape != truth_zyx.shape:
    raise ValueError(f'Reconstruction shape {recon_cm_inv.shape} != GT z,y,x {truth_zyx.shape}. Check TIGRE version/geometry ordering.')


## 6. Compare orientation conventions

The saved gVXR detector up-vector and +z scene rotation may differ from TIGRE’s sign or axis orientation. This cell evaluates the eight simple volume flips and both angle signs as a **diagnostic**. It does not silently choose and overwrite the reconstruction. Use the asymmetric-object case and inspect the plots before selecting a convention. A flip of projection detector columns/rows requires rerunning FDK with that flipped projection array.


In [ ]:
from itertools import product
from scipy.ndimage import binary_erosion
import pandas as pd

# Compare reconstructions for each angle sign and axis flip.
# Use the plots and saved geometry to choose the convention.
orient_rows = []
recon_candidates = {}
for sign in (1.0, -1.0):
    r = algs.fdk(proj, geo, ang * sign).astype(np.float32) * 10.0
    for flips in product((False, True), repeat=3):
        rr = np.flip(r, axis=tuple(i for i, f in enumerate(flips) if f)) if any(flips) else r
        rmse = float(np.sqrt(np.mean((rr - truth_zyx) ** 2)))
        key = (int(sign), flips)
        recon_candidates[key] = rr.copy()
        orient_rows.append({'angle_sign': int(sign), 'flip_z': flips[0], 'flip_y': flips[1],
                            'flip_x': flips[2], 'RMSE_cm_inv': rmse})
orientation_table = pd.DataFrame(orient_rows).sort_values('RMSE_cm_inv').reset_index(drop=True)
display(orientation_table.head(8))
print('Treat the best row as a candidate only. Confirm it visually and with an asymmetric phantom.')


## 7. Metrics: per-material mean μ, RMSE, and Dice

Set the selected angle sign and reconstruction-axis flips after reviewing the orientation candidates. Metrics use the full aligned grid for RMSE; material means and Dice are computed per non-background label. Edge erosion reduces partial-volume effects. Dice classifies every voxel, including background, to its nearest ground-truth μ value so false positives count. This is a simple baseline; for very similar material μ values, use hand-chosen thresholds instead.


In [ ]:
# Set these after reviewing the orientation table and slices.
SELECTED_ANGLE_SIGN = 1
SELECTED_FLIPS_ZYX = (False, False, False)

selected = recon_candidates[(SELECTED_ANGLE_SIGN, SELECTED_FLIPS_ZYX)]
# Segment by assigning each voxel to its nearest stored μ value.
label_map = gt_meta.get('label_map', {})
material_labels = [int(k) for k in label_map.keys() if int(k) != 0]
if not material_labels:
    material_labels = [int(v) for v in np.unique(labels_zyx) if int(v) != 0]

metrics = []
rmse = float(np.sqrt(np.mean((selected - truth_zyx) ** 2)))
mu_values = np.array([float(np.mean(truth_zyx[labels_zyx == lab])) for lab in material_labels])
valid_labels = np.array(material_labels, dtype=np.int32)
# Include background so false positives count in Dice.
class_labels = np.concatenate(([0], valid_labels))
class_mu = np.concatenate(([0.0], mu_values))
nearest = class_labels[np.argmin(np.abs(selected[..., None] - class_mu[None, None, None, :]), axis=-1)]
for lab, target_mu in zip(material_labels, mu_values):
    mask = labels_zyx == lab
    eroded = binary_erosion(mask, iterations=2, border_value=0)
    if eroded.sum() < 10:
        eroded = mask
    measured_mu = float(np.mean(selected[eroded]))
    pred = nearest == lab
    denom = int(pred.sum() + mask.sum())
    dice = float(2 * np.logical_and(pred, mask).sum() / denom) if denom else 1.0
    metrics.append({'label': lab, 'material': label_map.get(str(lab), f'label_{lab}'),
                    'ground_truth_mean_mu_cm^-1': float(target_mu),
                    'reconstruction_mean_mu_cm^-1': measured_mu,
                    'mean_mu_error_percent': float(100 * (measured_mu-target_mu) / target_mu) if target_mu else np.nan,
                    'RMSE_full_volume_cm^-1': rmse, 'Dice': dice,
                    'voxels_used_for_mean': int(eroded.sum())})
metrics_df = pd.DataFrame(metrics)
display(metrics_df.set_index('label'))


## 8. Inspect slices and save results

Look for alignment, orientation, edge blur, cupping, and cone-beam artifacts. FDK is approximate; poor-looking peripheral slices do not automatically imply a broken forward model. If the object is off-center, the saved ground-truth origin and TIGRE `offOrigin` mapping need careful coordinate-frame confirmation.


In [ ]:
import matplotlib.pyplot as plt

zmid, ymid, xmid = (np.array(selected.shape) // 2).tolist()
fig, ax = plt.subplots(2, 3, figsize=(14, 8))
ax[0,0].imshow(proj[len(proj)//4, :, :], cmap='gray'); ax[0,0].set_title('gVXR attenuation projection')
ax[0,1].imshow(truth_zyx[zmid], cmap='magma'); ax[0,1].set_title('Ground truth μ, central z')
ax[0,2].imshow(selected[zmid], cmap='magma'); ax[0,2].set_title('FDK μ, central z')
ax[1,0].imshow(truth_zyx[:, ymid, :], cmap='magma'); ax[1,0].set_title('Ground truth μ, central y')
ax[1,1].imshow(selected[:, ymid, :], cmap='magma'); ax[1,1].set_title('FDK μ, central y')
ax[1,2].imshow(selected[zmid] - truth_zyx[zmid], cmap='coolwarm'); ax[1,2].set_title('FDK − GT, central z')
for a in ax.flat: a.axis('off')
plt.tight_layout(); plt.show()

np.save('tigre_reconstruction_cm_inv.npy', selected)
metrics_df.to_csv('tigre_validation_metrics.csv', index=False)
with open('tigre_validation_summary.json', 'w') as f:
    json.dump({'source_h5': h5_path, 'SOD_mm': SOD, 'SDD_mm': SDD,
               'magnification': mag, 'voxel_size_mm': voxel_mm,
               'angle_sign': SELECTED_ANGLE_SIGN, 'flips_zyx': SELECTED_FLIPS_ZYX,
               'global_RMSE_cm_inv': rmse, 'materials': metrics}, f, indent=2)
print('Saved: tigre_reconstruction_cm_inv.npy, tigre_validation_metrics.csv, tigre_validation_summary.json')


## Important interpretation notes

- The reconstruction is only quantitatively comparable for monochromatic clean data; beam hardening and detector effects change the projection model.
- The saved gVXR ground-truth μ volume is computed by the project with xraylib at the stored energy, and uses cm⁻¹. It is a synthetic reference, not experimental validation.
- TIGRE is an independent reconstruction implementation, but geometry/orientation must be calibrated first. Report angle sign, detector flips, `offOrigin`, and reconstruction settings with your results.
- The simple nearest-μ Dice segmentation is a baseline. For multi-material scenes, use explicit thresholds or a calibrated classifier, and erode masks only for mean-μ calculations.
- TIGRE installation/reconstruction requires CUDA and an NVIDIA GPU. The project’s gVXR EGL/OpenGL GPU status is separate from TIGRE’s CUDA support.
